#### 4.6. Know your data!

What does our data tells us about itself? Lets check some things. 

Lets check what does clerc data contain

In [1]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

First we clearly have an imbalance between the amout of queries and the text corpus that we actually store. This is not a bad thing its actually means we have a lot of false positives to run againts. 

However, if you will look closely how we we contruct corpora, it becomes obvious that was by design: 
```python
for p in (*row["positive_passages"], *row["negative_passages"]):
            did = p["docid"]
            if did not in corpus:
                corpus[did] = Document(doc_id=did, text=p["text"])
```

Moreover things that we store in our collection as `Points` are not actual documents - they are passages from some document. Our limit is 2000 - meaning we have exactly 2000 documents in our dataset. That means what we might have suspected before - the documents are already chunked, and with the chunking we get all the complications that it entails.

##### How chunked exactly?

If the corpus is pre-chunked, the passage lengths should betray it: a chunker cuts at a fixed budget, so lengths pile up at the limit instead of spreading out the way natural documents do. Whole federal court opinions run thousands of words — let's see what we actually have.

In [2]:
import numpy as np

word_counts = np.array([len(p.text.split()) for p in clerc_data.corpus.values()])
print(
    f"words/passage: min={word_counts.min()}  median={int(np.median(word_counts))}  max={word_counts.max()}\n"
    f"passages at exactly {word_counts.max()} words: {(word_counts == word_counts.max()).mean():.1%}\n" 
    f'stadard deviation is {np.std(word_counts)}'
)

words/passage: min=153  median=350  max=350
passages at exactly 350 words: 98.0%
stadard deviation is 10.981116005085166


The distribution is a wall at 350 words — the signature of a fixed-size chunker, not of natural documents. The distribution of the size difference is just around 10 words - means most of they are actually 350 words.

And the CLERC repo indeed ships both granularities side by side:
- `collection/collection.doc.tsv.gz` — full documents (**dids**)
- `collection/collection.passage.tsv.gz` — the ≤350-word chunks (**pids**)
- `collection/mapping.pid2did.tsv` — which document each passage was cut from

The `docid` fields we have been treating as document ids are actually **passage ids** (the name comes from the Tevatron training format, where every retrieval unit is generically a "document"). The mapping file lets us ask the question this whole section is about: how many passages in our pool are *siblings* — different slices of the same underlying case?

This matters because our qrels are passage-level. If the pool contains other slices of the very document a query cites, a retriever can find the right case and still score zero. And since the actual task is "find the case to cite", those labels act as **false negatives**: relevant at the document level, labeled irrelevant at the passage level.

In [3]:
from src.doc_mapping import load_pid2did, analyze_siblings

pid2did = load_pid2did(clerc_data.corpus)
sibling_stats = analyze_siblings(clerc_data, pid2did)
print(sibling_stats)

Sibling structure (41,167 passages, 2,000 queries):
  Unique source documents:               23,221
  Docs with >1 passage in pool:          9,442 (40.7%)
  Queries w/ positive siblings in pool:  435 (21.8%)
  Queries w/ same-doc hard negatives:    254 (12.7%)


Three facts fall out:

1. **Our 41,167 "documents" are passages from only 23,221 cases** — over 40% of the cases contribute more than one passage to the pool. The pool is far more entangled than the flat corpus suggested.
2. **~22% of queries have siblings of their positive in the pool.** For those queries, "right case, wrong slice" is a possible outcome — and it scores exactly zero.
3. **~13% of queries have passages of the correct case mined as their own hard negatives.** This is the sharpest version of the problem: the labels actively penalize retrieving the cited case unless the retriever picks the one blessed ≤350-word slice among up to 8 near-identical siblings.

Fact 3 reframes our reranker results. A cross-encoder judging "is this passage about the same matter as the query?" will confidently promote sibling passages — and every sibling it lifts above the labeled positive costs NDCG. The better the model is at *document*-level relevance, the harder the passage-level qrels punish it.

Let's look at one of these queries up close.

In [3]:
ex = max(sibling_stats.examples, key=lambda e: len(e.sibling_negative_pids))
print(f"query {ex.query_id}:\n{clerc_data.queries[ex.query_id].query[:400]}...\n")

pos_pid = ex.positive_pids[0]
print(f"POSITIVE   pid={pos_pid}  did={pid2did[pos_pid]}")
print(clerc_data.corpus[pos_pid].text[:300], "...\n")

for pid in ex.sibling_negative_pids[:3]:
    print(f'"NEGATIVE"  pid={pid}  did={pid2did[pid]}')
    print(clerc_data.corpus[pid].text[:300], "...\n")

query 704911:
"512(k)(1)(A)]."" 17 U.S.C. § 512(k)(1)(B). ""Service provider” thus is defined more narrowly with respect to the ""conduit” safe harbor provision. . The parties do not dispute that Hurricane, OPG; and Swarthmore had valid section 512(i) policies. See, e.g., Complaint, p. 5:20-23 & Ex. D (email from Ralph E. Jocke), although there is no evidence in the record as to this point with respect to OPG a...

POSITIVE   pid=3727252  did=11422779
an interest in the content of that data.” Id. at 18 n. 19. The inclusion of section 512(d) which creates a “safe harbor” for copyright infringement resulting from the use of information location tools by service providers, which include directories, indexes, references, pointers and hypertext links, ...

"NEGATIVE"  pid=3727249  did=11422779
II of the Digital Millennium Copyright Act (“DMCA”), Pub. L.. 105-304, Title II, § 202(a), 112 Stat. 2877 (1998)(codified at 17 U.S.C. § 512). The DMCA marked Congress’ entry into the online copyright

#### What this changes

Nothing about our *relative* comparisons is broken — every system faced the same pool and the same labels. But the interpretation of the earlier sections shifts:

- **Chunking as an improvement strategy was off the table before we even started.** The dataset had already chunked for us, at ≤350 words, mid-document. Our chunking analysis in section 4.5 was measuring chunk sizes and calling them document sizes — its conclusion (don't chunk) was right, but by construction, not by analysis.
- **Hybrid and reranker underperformance is partly a labeling artifact.** Passage-level qrels give zero credit for "right case, wrong slice", and for ~13% of queries they actively punish it via same-document hard negatives.
- **Absolute numbers are not comparable to the CLERC paper**, which retrieves against the full ~23.7M-passage collection rather than a pooled ~41k candidates.

The meta-lesson for your own project: this section belongs *before* the baseline, not after the third reranker. Every dataset encodes decisions someone else made — chunking budgets, negative mining, labeling granularity — and your metrics silently inherit all of them.

In [4]:
from qdrant_client import QdrantClient
import os

qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

In [5]:
# constants
COLLECTION_NAME = 'lawyer_citations'
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SIZE = 384

In [6]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance


config = EmbeddingConfig(
    name='dense_base',
    model_id=DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
    providers=["CoreMLExecutionProvider", "CPUExecutionProvider"],
    parallel=None,
)

In [7]:
from src.search import DenseSearcher

searcher = DenseSearcher(client, COLLECTION_NAME, config)

In [13]:
from src.evaluation import evaluate_retrieval_graded

report = evaluate_retrieval_graded(searcher, clerc_data, pid2did)  # smoke run
print(report)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1380
  recall@10    0.2594
  mrr@10       0.1143


In [7]:

DENSE_MODEL_V2  = 'BAAI/bge-base-en-v1.5' #  0.320 GB,	768 DIM	
DENSE_SIZE_V2 = 768
COLLECTION_NAME_V2 = 'trec_dl_v2'
SPARSE_MODEL_V2 = 'Qdrant/bm25' 

In [8]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance
from qdrant_client.models import Modifier


configs_v2 = [
    EmbeddingConfig(
        name='dense_base',
        model_id=DENSE_MODEL_V2,
        kind='dense',
        backend='sentence-transformers',
        size=DENSE_SIZE_V2,
        distance=Distance.COSINE,
    ),
    EmbeddingConfig(
        name="bm25",
        model_id=SPARSE_MODEL_V2,
        kind="sparse",
        modifier=Modifier.IDF
    )
]

In [9]:
from src.search import HybridSearcher

searcher_v2 = HybridSearcher(
    client, 
    COLLECTION_NAME_V2, 
    configs_v2,
)

In [17]:
from src.evaluation import evaluate_retrieval_graded

report = evaluate_retrieval_graded(searcher_v2, clerc_data, pid2did)  # smoke run
print(report)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1647
  recall@10    0.2861
  mrr@10       0.1369


In [10]:
from src.evaluation import diagnose_doc_level

diagnosis = diagnose_doc_level(searcher_v2, clerc_data, pid2did)
print(diagnosis)

diagnose:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Document-level diagnosis (2000 queries, top-10):
  doc hit rate (any slice of positive doc): 696 (34.8%)
    exact positive retrieved:               674 (33.7%)
    sibling only (right doc, wrong slice):  22 (1.1%)
  full miss (document absent):              1304 (65.2%)
  mean doc precision@10:          0.051


In [11]:
import os

os.cpu_count()

15